# Section 1b — Compute node metrics locally
### Replaces the Gephi export entirely

Gephi's *Export table* can silently produce an empty file (usually because a **Filter** is
still active — it exports only the visible rows). Rather than depend on it, this notebook
computes every metric in Python and writes `data/gephi_node_metrics.csv` in exactly the
format `section1_sna_analysis.ipynb` expects.

Two practical advantages:
- **Speed** — sampled betweenness takes ~11s here vs 10–30 minutes for Gephi's exact Network Diameter
- **Reproducibility** — a seeded script beats GUI clicks you cannot re-run

Gephi is still worth using for the **visualisation**. This just removes it from the
critical path for your numbers.

In [1]:
import pandas as pd, numpy as np, networkx as nx, random, time
from pathlib import Path
Path("data").mkdir(exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

edges = pd.read_csv("data/gephi_edges.csv")
nodes = pd.read_csv("data/gephi_nodes.csv")
G = nx.DiGraph()
for r in edges.itertuples(index=False):
    G.add_edge(r.Source, r.Target, weight=int(r.Weight))
U = G.to_undirected()
label = dict(zip(nodes.Id, nodes.Label))
print(f"nodes {G.number_of_nodes():,} | edges {G.number_of_edges():,}")

nodes 6,434 | edges 259,122


## Louvain community detection

Same algorithm Gephi's *Modularity* runs. Compare the Q score and community count against
what Gephi gave you — they should be close (Louvain is stochastic, so not identical).

In [2]:
t=time.time()
comms = nx.community.louvain_communities(U, weight="weight", seed=SEED, resolution=1.0)
Q = nx.community.modularity(U, comms, weight="weight")
print(f"communities : {len(comms)}")
print(f"modularity Q: {Q:.4f}   ({time.time()-t:.1f}s)")

sizes = sorted((len(x) for x in comms), reverse=True)
print(f"sizes       : {sizes[:12]}")
print(f">=20 nodes  : {sum(1 for s in sizes if s>=20)} communities")

comm_of = {}
for i, grp in enumerate(comms):
    for v in grp: comm_of[v] = i

print("\nINTERPRETATION")
if Q < 0.3:
    print(f"  Q = {Q:.3f} is LOW. Below ~0.3 means WEAK community structure -")
    print("  the network is one densely interconnected mesh rather than separate groups.")
    print("  Report this honestly; it is a real property of the network, not an error.")
else:
    print(f"  Q = {Q:.3f} indicates meaningful community structure.")

communities : 10
modularity Q: 0.1253   (3.0s)
sizes       : [1587, 1227, 1102, 912, 738, 635, 115, 95, 19, 4]
>=20 nodes  : 8 communities

INTERPRETATION
  Q = 0.125 is LOW. Below ~0.3 means WEAK community structure -
  the network is one densely interconnected mesh rather than separate groups.
  Report this honestly; it is a real property of the network, not an error.


## Centralities

Betweenness and closeness are approximated by sampling sources — exact computation is
O(n·m) ≈ 1.7 billion operations here. **State the sampling in your methodology.**

In [3]:
t=time.time(); pr = nx.pagerank(G, weight="weight"); print(f"pagerank      {time.time()-t:5.1f}s")
t=time.time(); ev = nx.eigenvector_centrality_numpy(U, weight="weight"); print(f"eigenvector   {time.time()-t:5.1f}s  (undirected: directed version is ambiguous with 25 SCCs)")
t=time.time(); core = nx.core_number(U); print(f"core number   {time.time()-t:5.1f}s")

K_BTW = 300
t=time.time(); btw = nx.betweenness_centrality(G, k=K_BTW, seed=SEED, normalized=True)
print(f"betweenness   {time.time()-t:5.1f}s  (sampled, k={K_BTW})")

K_CLO = 600
samp = random.sample(list(U.nodes()), K_CLO)
t=time.time()
clo = {v: nx.closeness_centrality(U, u=v) for v in samp}
print(f"closeness     {time.time()-t:5.1f}s  (sampled, {K_CLO} nodes; others NaN)")

pagerank        0.2s
eigenvector     0.2s  (undirected: directed version is ambiguous with 25 SCCs)
core number     0.2s
betweenness     9.9s  (sampled, k=300)
closeness       5.8s  (sampled, 600 nodes; others NaN)


## Write the metrics file

Column names deliberately match Gephi's so `section1_sna_analysis.ipynb` works unchanged.

In [4]:
df = pd.DataFrame({"Id": list(G.nodes())})
df["Label"]                   = df.Id.map(label)
df["Modularity Class"]        = df.Id.map(comm_of)
df["PageRank"]                = df.Id.map(pr)
df["Eigenvector Centrality"]  = df.Id.map(ev)
df["Betweenness Centrality"]  = df.Id.map(btw)
df["Closeness Centrality"]    = df.Id.map(clo)
df["Core Number"]             = df.Id.map(core)
df["In-Degree"]               = df.Id.map(dict(G.in_degree()))
df["Out-Degree"]              = df.Id.map(dict(G.out_degree()))
df["Degree"]                  = df["In-Degree"] + df["Out-Degree"]

df.to_csv("data/gephi_node_metrics.csv", index=False)
print(f"wrote data/gephi_node_metrics.csv  ->  {len(df):,} rows x {len(df.columns)} cols")
print("columns:", list(df.columns))
display(df.head())

assert len(df) == 6434, "row count wrong"
print("\n>>> now re-run the Block C and Block D cells in section1_sna_analysis.ipynb")

wrote data/gephi_node_metrics.csv  ->  6,434 rows x 11 cols
columns: ['Id', 'Label', 'Modularity Class', 'PageRank', 'Eigenvector Centrality', 'Betweenness Centrality', 'Closeness Centrality', 'Core Number', 'In-Degree', 'Out-Degree', 'Degree']


,Id,Label,Modularity Class,PageRank,Eigenvector Centrality,Betweenness Centrality,Closeness Centrality,Core Number,In-Degree,Out-Degree,Degree
0,u_44176596eca7,user_01999,5,0.000089,0.004163,0.000104,NaN,31,26,24,50
1,u_d57f77166952,user_00282,0,0.000547,0.033481,0.000920,NaN,84,141,167,308
2,u_f6454d51902e,user_01647,1,0.000206,0.009535,0.000287,NaN,66,54,59,113
3,u_e1686e72fc65,user_00750,0,0.000406,0.019583,0.000761,0.469014,84,137,79,216
4,u_95cb88ab48b4,user_02547,0,0.000085,0.002449,0.000035,0.377856,24,21,18,39



>>> now re-run the Block C and Block D cells in section1_sna_analysis.ipynb


## Community summary table

Per-community size and internal density — goes straight into the report.

In [5]:
rows=[]
m = G.number_of_edges()
intra = sum(1 for u,v in G.edges() if comm_of.get(u)==comm_of.get(v))
print(f"intra-community edges: {intra:,} ({100*intra/m:.1f}%)")
print(f"inter-community edges: {m-intra:,} ({100*(m-intra)/m:.1f}%)")

for i,grp in enumerate(comms):
    if len(grp)<20: continue
    sub=G.subgraph(grp)
    rows.append({"community":i,"nodes":len(grp),"edges":sub.number_of_edges(),
                 "density":round(nx.density(sub),5),
                 "avg_degree":round(2*sub.number_of_edges()/len(grp),2)})
cdf=pd.DataFrame(rows).sort_values("nodes",ascending=False)
display(cdf)
cdf.to_csv("data/community_summary.csv", index=False)
print("saved data/community_summary.csv")

intra-community edges: 75,469 (29.1%)
inter-community edges: 183,653 (70.9%)


,community,nodes,edges,density,avg_degree
5,5,1587,19533,0.00776,24.62
6,6,1227,19792,0.01316,32.26
3,3,1102,18849,0.01554,34.21
0,0,912,7047,0.00848,15.45
4,4,738,4434,0.00815,12.02
1,1,635,5162,0.01282,16.26
7,8,115,336,0.02563,5.84
2,2,95,273,0.03057,5.75


saved data/community_summary.csv
